In [ ]:
import sys
from pathlib import Path

# To import the variables in config.py
PROJECT_ROOT = Path.cwd().parent
sys.path.append(str(PROJECT_ROOT / "src"))

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as f 
from pyspark.sql.types import DoubleType
from config import OFF_CSV_PATH, OFF_CSV_URL
import urllib.request

In [3]:
spark = SparkSession.builder \
.appName("lakehouse-off") \
.master("local[*]") \
.config("spark.driver.memory", "2g") \
.config("spark.sql.shuffle.partitions", "4") \
.config("spark.executor.memory", "1g") \
.getOrCreate() 

#### Read Dataset from https url

In [4]:
if not OFF_CSV_PATH.exists():
    # Create the data/raw dir
    OFF_CSV_PATH.parent.mkdir(parents=True, exist_ok=True)
    # Download the csv file only when it doesn't exist 
    urllib.request.urlretrieve(OFF_CSV_URL, OFF_CSV_PATH)

In [5]:
# Verify the file path and its size. 
print(OFF_CSV_PATH.exists())
print(OFF_CSV_PATH.stat().st_size)

True
1261161707


#### Read the whole data set

In [6]:
df = spark.read.csv(
    str(OFF_CSV_PATH),
    sep = "\t",
    header=True,
    multiLine=True,
    quote='"',
    escape='"',
    encoding="utf-8"
    )

In [7]:
df.show(10)

+------------+--------------------+--------------------+----------+--------------------+---------------+----------------------+----------------+--------------+---------------------+--------------------+------------------------+------------+--------+---------+--------------+------------+--------------+--------------+-----------------+--------------+--------------------+--------------------+--------------------+-------+------------+----------+--------------------+-------------------------+--------------------+--------------------+--------------------+---------+--------------+------------------------+------+-----------+---------------+------+----------------+-------------------+-------------+--------------------+--------------------+-------------------------+---------+------------+------+-----------+---------+------------+----------------+-----------------+-----------+---------+---------------+--------------------+----------------+----------------+----------+-------------+----------------

In [8]:
df.printSchema()

root
 |-- code: string (nullable = true)
 |-- url: string (nullable = true)
 |-- creator: string (nullable = true)
 |-- created_t: string (nullable = true)
 |-- created_datetime: string (nullable = true)
 |-- last_modified_t: string (nullable = true)
 |-- last_modified_datetime: string (nullable = true)
 |-- last_modified_by: string (nullable = true)
 |-- last_updated_t: string (nullable = true)
 |-- last_updated_datetime: string (nullable = true)
 |-- product_name: string (nullable = true)
 |-- abbreviated_product_name: string (nullable = true)
 |-- generic_name: string (nullable = true)
 |-- quantity: string (nullable = true)
 |-- packaging: string (nullable = true)
 |-- packaging_tags: string (nullable = true)
 |-- packaging_en: string (nullable = true)
 |-- packaging_text: string (nullable = true)
 |-- brands: string (nullable = true)
 |-- brands_tags: string (nullable = true)
 |-- brands_en: string (nullable = true)
 |-- categories: string (nullable = true)
 |-- categories_tags: s

In [9]:
print(f"There are {len(df.columns)} columns in the whole data set.")
print(f"There are {df.count()} rows in the whold data set.")

There are 210 columns in the whole data set.
There are 4475958 rows in the whold data set.


In [10]:
# Check the columns contains 'nutrition_100g'
numeric_cols = [col for col in df.columns if "100g" in col]
len(numeric_cols)
print(numeric_cols)

['energy-kj_100g', 'energy-kcal_100g', 'energy_100g', 'energy-from-fat_100g', 'fat_100g', 'saturated-fat_100g', 'butyric-acid_100g', 'caproic-acid_100g', 'caprylic-acid_100g', 'capric-acid_100g', 'lauric-acid_100g', 'myristic-acid_100g', 'palmitic-acid_100g', 'stearic-acid_100g', 'arachidic-acid_100g', 'behenic-acid_100g', 'lignoceric-acid_100g', 'cerotic-acid_100g', 'montanic-acid_100g', 'melissic-acid_100g', 'unsaturated-fat_100g', 'monounsaturated-fat_100g', 'omega-9-fat_100g', 'polyunsaturated-fat_100g', 'omega-3-fat_100g', 'omega-6-fat_100g', 'alpha-linolenic-acid_100g', 'eicosapentaenoic-acid_100g', 'docosahexaenoic-acid_100g', 'linoleic-acid_100g', 'arachidonic-acid_100g', 'gamma-linolenic-acid_100g', 'dihomo-gamma-linolenic-acid_100g', 'oleic-acid_100g', 'elaidic-acid_100g', 'gondoic-acid_100g', 'mead-acid_100g', 'erucic-acid_100g', 'nervonic-acid_100g', 'trans-fat_100g', 'cholesterol_100g', 'carbohydrates_100g', 'sugars_100g', 'added-sugars_100g', 'sucrose_100g', 'glucose_100g

#### Exploring data by creating a sample 

In [11]:
# create a 1% sample DataFrame randomly
df_sample = df.sample(fraction=0.01, seed=42)
total_sample = df_sample.count()

##### Sample data: Check the Null values in column level 

In [12]:
# Before counting Null values, make sure to replace some string values like "unknown", "N/A" or "" values into NULL value
def replace_missing_values(df):
    string_cols = [c for c, t in df.dtypes if t == "string"]

    return df.select([
        f.when(
            f.trim(f.lower(f.col(c))).isin("unknown", "n/a", "na", ""),
            None
        ).otherwise(f.col(c)).alias(c)
        if c in string_cols else f.col(c)
        for c in df.columns
    ])
    

In [13]:
df_sample_1 = replace_missing_values(df_sample)

In [14]:
# Check the percentage of Null values in each column and pivot the DataFrame 
null_rates = df_sample_1.select([
    f.round(
        (f.count(f.when(f.col(c).isNull(), 1)) / total_sample), 2
    ).alias(c) for c in df_sample.columns
])

df_null = null_rates.select(
    f.explode(
        f.array(
            [f.struct(
                    f.lit(c).alias("col_name"),
                    f.col(c).alias("null_rate")
                ) for c in null_rates.columns]
        )
    ).alias("tmp")
).select(
    "tmp.col_name",
    "tmp.null_rate"
    ).orderBy(
        f.col("null_rate").desc())

In [15]:
df_null.show(210, truncate=False)

+--------------------------------+---------+
|col_name                        |null_rate|
+--------------------------------+---------+
|cities                          |1.0      |
|allergens_en                    |1.0      |
|no_nutrition_data               |1.0      |
|additives                       |1.0      |
|energy-from-fat_100g            |1.0      |
|butyric-acid_100g               |1.0      |
|caproic-acid_100g               |1.0      |
|caprylic-acid_100g              |1.0      |
|capric-acid_100g                |1.0      |
|lauric-acid_100g                |1.0      |
|myristic-acid_100g              |1.0      |
|palmitic-acid_100g              |1.0      |
|stearic-acid_100g               |1.0      |
|arachidic-acid_100g             |1.0      |
|behenic-acid_100g               |1.0      |
|lignoceric-acid_100g            |1.0      |
|cerotic-acid_100g               |1.0      |
|montanic-acid_100g              |1.0      |
|melissic-acid_100g              |1.0      |
|unsaturat

##### Trim columns 

After inspecting the null values existing in the sample, it's better to select the useful columns rather than to drop the uesless ones. 

In [16]:
def trim_columns_and_drop_product_code_null(df):

    cols_to_keep = [
        "code",
        "last_modified_t",
        "product_name",
        "brands",
        "categories_en",
        "labels_en",
        "countries_en",
        "nutriscore_grade",
        "ingredients_tags",
        "food_groups_en",
        "energy-kcal_100g",
        "fat_100g",
        "saturated-fat_100g",
        "carbohydrates_100g",
        "sugars_100g",
        "proteins_100g",
        "salt_100g"
    ]
    return df.select([c for c in cols_to_keep if c in df.columns]).filter(
            f.col("product_name").isNotNull()
            & f.col("code").isNotNull()
            & (f.col("product_name") != "")
            & (f.col("code") != ""))

In [17]:
df_sample_trimmed = trim_columns_and_drop_product_code_null(df_sample_1)

In [18]:
df_sample_trimmed.show(5)

+-------------+---------------+--------------------+--------------------+--------------------+--------------------+-------------+----------------+----------------+--------------+----------------+--------+------------------+------------------+-----------+-------------+---------+
|         code|last_modified_t|        product_name|              brands|       categories_en|           labels_en| countries_en|nutriscore_grade|ingredients_tags|food_groups_en|energy-kcal_100g|fat_100g|saturated-fat_100g|carbohydrates_100g|sugars_100g|proteins_100g|salt_100g|
+-------------+---------------+--------------------+--------------------+--------------------+--------------------+-------------+----------------+----------------+--------------+----------------+--------+------------------+------------------+-----------+-------------+---------+
|     00000269|     1728035306|Bulletproof colla...|         Bulletproof|Collagen,Collagen...|                NULL|      Ireland|            NULL|            NULL|

##### Check the outlier in Nutritional Columns 

In [19]:
# Cast Nutritional columns : StringType -> DoubleType
Nutri_cols = [
    "energy-kcal_100g",
    "fat_100g",
    "saturated-fat_100g",
    "carbohydrates_100g",
    "sugars_100g",
    "proteins_100g",
    "salt_100g"
]

for col in Nutri_cols:
    df_sample_trimmed = df_sample_trimmed.withColumn(col, f.col(col).try_cast(DoubleType()))

In [20]:
# Check the outiler with approximate quantiles:
proba = [0.25, 0.5, 0.75, 0.95, 0.99]
for col in Nutri_cols:
    quantiles = df_sample_trimmed.filter(f.col(col).isNotNull()).approxQuantile(col, proba, 0.01)
    print(f"\n{col}")
    for p,v in zip(proba, quantiles):
        print(f"p{int(p*100):02d}: {v:.3f}")


energy-kcal_100g
p25: 109.000
p50: 250.000
p75: 389.000
p95: 562.500
p99: 31500.000

fat_100g
p25: 1.250
p50: 7.300
p75: 20.700
p95: 42.000
p99: 1300.000

saturated-fat_100g
p25: 0.200
p50: 1.800
p75: 6.800
p95: 19.000
p99: 450.000

carbohydrates_100g
p25: 2.800
p50: 14.000
p75: 50.000
p95: 75.000
p99: 1300.000

sugars_100g
p25: 0.500
p50: 3.200
p75: 13.000
p95: 52.000
p99: 340.000

proteins_100g
p25: 2.000
p50: 6.667
p75: 13.333
p95: 26.000
p99: 3500.000

salt_100g
p25: 0.100
p50: 0.580
p75: 1.377
p95: 3.520
p99: 2166.667


In [21]:
# as the output of p99 are extremly high, so we need to filter the outliers 
def filter_outiler_nutrition(df):
    for col in Nutri_cols:
        df = df.withColumn(col, f.col(col).try_cast(DoubleType()))

    df_full_outlier_filtered = df.filter(
        (f.col("energy-kcal_100g").isNull() | f.col("energy-kcal_100g").between(0, 900)) &
        (f.col("fat_100g").isNull() | f.col("fat_100g").between(0, 100)) &
        (f.col("saturated-fat_100g").isNull() | f.col("saturated-fat_100g").between(0, 100)) &
        (f.col("carbohydrates_100g").isNull() | f.col("carbohydrates_100g").between(0, 100)) &
        (f.col("sugars_100g").isNull() | f.col("sugars_100g").between(0, 100)) &
        (f.col("proteins_100g").isNull() | f.col("proteins_100g").between(0, 100)) &
        (f.col("salt_100g").isNull() | f.col("salt_100g").between(0, 100))
    )
    return df_full_outlier_filtered

#### In the full dataset

In [22]:
df_full_replaced = replace_missing_values(df)
df_full_trimmed = trim_columns_and_drop_product_code_null(df_full_replaced)
df_full_outlier_filtered = filter_outiler_nutrition(df_full_trimmed)

##### Check the Null values in row level 

In [23]:
col_to_check = [c for c in df_full_outlier_filtered.columns if c not in ["product_name", "code"]]
nb_col = len(col_to_check)

df_full_rows_null = df_full_outlier_filtered.withColumn(
        "null_count",
        sum(f.col(c).isNull().try_cast("int") for c in col_to_check)
    ).withColumn(
        "null_perc",
        f.round(f.col("null_count") / nb_col, 2)
    ).orderBy(f.col("null_perc").desc())

In [24]:
# Remove the rows which percentage of null values are 100% 
df_full_rows_clean = df_full_rows_null.filter(f.col("null_perc") < 1)
total_rows = df_full_rows_clean.count()
print(f"There are {total_rows} rows in the full dataset after removing full null value rows.")

There are 4136243 rows in the full dataset after removing full null value rows.


##### Duplication Check in *Product Code*

In [25]:
distinct_codes = df_full_rows_clean.select("code").distinct().count()
dup_count = total_rows - distinct_codes 

print(f"Total rows: {total_rows:,}")
print(f"Distinct codes: {distinct_codes:,}")
print(f"Duplicate rows: {dup_count:,} ({dup_count/total_rows:.1%})")

Total rows: 4,136,243
Distinct codes: 4,136,190
Duplicate rows: 53 (0.0%)


# 

In [28]:
df_full_clean = df_full_rows_clean

In [ ]:
# # For all the duplicated codes, verify the modification time, from oldest to latest
# impor os 
# checkpoint_dir = str(PROJECT_ROOT/"data"/"checkpoints")
# checkpoint_dir = os.makedirs(checkpoint_dir, exist_ok=True)
# spark.sparkContext.setCheckpointDir(checkpoint_dir)

# if int(dup_count) > 0:
#     df_full_rows_clean.groupBy("code") \
#     .agg(
#         f.count("*").alias("count_dup"), 
#         f.min("last_modified_t").alias("oldest_t"), 
#         f.max("last_modified_t").alias("latest_t")
#     ) \
#     .filter(f.col("count_dup") > 1) \
#     .orderBy(f.col("count_dup").desc()) \
#     .show(10, truncate=False)
#     # Use .checkpoint() because lineage is long and RAM no-sufficient
#     df_full_clean = df_full_rows_clean.dropDuplicates(["code"]).checkpoint(eager=False)
#     df_full_clean.count()
# else: 
#     df_full_clean = df_full_rows_clean.checkpoint()
#     df_full_clean.count()

##### Categorical distributions

In [29]:
Cat_cols = [
    "categories_en",
    "labels_en",
    "countries_en",
    "nutriscore_grade",
    "ingredients_tags",
    "food_groups_en"
]

# Distinct value counts
df_full_clean.agg(
    *[f.countDistinct(col).alias(col) for col in Cat_cols]
).show()

# nutriscore_grade distinction
df_full_clean.groupBy(f.col("nutriscore_grade"))\
    .count() \
    .orderBy(f.col("count").desc()) \
    .show()

# Top 10 brands and categories by product count
for col_name in  ["categories_en", "brands"]:
    df_full_clean.groupBy(f.col(col_name)) \
    .count() \
    .orderBy(f.col("count").desc()) \
    .show(10, truncate=False)

+-------------+---------+------------+----------------+----------------+--------------+
|categories_en|labels_en|countries_en|nutriscore_grade|ingredients_tags|food_groups_en|
+-------------+---------+------------+----------------+----------------+--------------+
|       182673|   128634|        8876|               8|          899791|            48|
+-------------+---------+------------+----------------+----------------+--------------+

+----------------+-------+
|nutriscore_grade|  count|
+----------------+-------+
|            NULL|2702135|
|               e| 374906|
|               d| 336709|
|               c| 277918|
|               a| 195245|
|               b| 154375|
|  not-applicable|  94953|
|           Italy|      1|
|             590|      1|
+----------------+-------+

+--------------------------------------------------------------------------------------------------------+-------+
|categories_en                                                                              

In [ ]:
# # Transform anormal values in "nutriscore_grade" into None
# valid_grades = ["a", "b", "c", "d", "e"]

# df_full_clean = df_full_clean.withColumn(
#     "nutriscore_grade",
#     f.when(
#         f.lower(f.col("nutriscore_grade")).isin(valid_grades),
#         f.lower(f.col("nutriscore_grade"))
#     ).otherwise(None)
# )

#### Time Range of *last_modified_t* in the full dataset

In [ ]:
# Check the time range of records in the full dataset: if most of records have same timestamp -> data was dumped  
df_full_clean.select(
    f.from_unixtime(f.min("last_modified_t")).alias("min_date"),
    f.from_unixtime(f.max("last_modified_t")).alias("max_date")
).show()

# Get the year distribution to know if the dataset is a historical snapshot or not -> incremental load or not 
df_full_clean.withColumn(
    "Year",
    f.year(f.from_unixtime(f.col("last_modified_t").cast("long")))
).groupBy("Year").count().orderBy("Year").show()

+-------------------+-------------------+
|           min_date|           max_date|
+-------------------+-------------------+
|2013-01-06 18:05:47|2026-04-29 16:28:35|
+-------------------+-------------------+

+----+-------+
|Year|  count|
+----+-------+
|2013|      3|
|2014|     14|
|2015|    242|
|2016|    946|
|2017|  12999|
|2018|   6669|
|2019|  32949|
|2020| 214011|
|2021|  71522|
|2022| 216915|
|2023| 243392|
|2024| 337798|
|2025| 586029|
|2026|2412701|
+----+-------+



So according to the max et min timestamp, the max value of *last_modified_t* can be used as watermark. Each pipeline run just need to track the max timestamp from the previous run.

#### Check the data format in *ingredients_tags*

In [ ]:
df_full_clean.filter(f.col("ingredients_tags").isNotNull()) \
    .select("ingredients_tags") \
    .limit(10) \
    .show(truncate=False)

+-----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|ingredients_tags                                                                                                                     

In [ ]:
# Transform the format into ArrayType
df_full_clean = df_full_clean.withColumn(
    "ingredients_tags",
    f.transform(
        f.split(f.col("ingredients_tags"), ","),
        lambda x: f.regexp_replace(f.trim(x), "^[a-z]{2}:", "")
    )
)